# Caso de Estudio: Modelado y generación de insights en la industria azucarera

**Ingenio Providencia · Aprendizaje Automático II · Prof. Anibal Sosa**

**Autores:** 
- Angie Gisselle Perez Ramirez
- Jose Ricardo Florez leon
- Wilson Andrés Panesso Barbosa
- Michael Alvarado Hernández

 · **Fecha:** noviembre de 2026

Este notebook contiene todo el modelado del caso (Tarea I). Parte del análisis exploratorio y del
preprocesamiento que ya se hicieron en el taller anterior, y va de lo simple a lo complejo:

| Sección | Qué hace | Punto del enunciado |
|---|---|---|
| 0 | Configuración | — |
| 1 | Datos, targets y partición 80/20 | 2.3 |
| 2 | Preprocesamiento dentro de un `Pipeline` | 2.3 |
| 3 | Modelos de referencia de regresión (lineal múltiple) | 2.1 |
| 4 | Modelos de referencia de clasificación (logística multinomial y KNN) | 2.1 |
| 5 | Modelos avanzados de regresión | 2.2 |
| 6 | Modelos avanzados de clasificación | 2.2 |
| 7 | Comparación y selección del mejor modelo | 2.3 |
| 8 | Interpretabilidad e insights para el Ingenio | 2.2 |
| 9 | Conclusiones | — |

**Problemas que se resuelven**

- **Regresión** (HISTORICO_SUERTES): predecir las toneladas de caña por hectárea (**TCH**) y el porcentaje de
  sacarosa (**%Sac.Caña**).
- **Clasificación** (BD_IPSA_1940): clasificar el **TCH** y la **sacarosa** en niveles Bajo, Medio y Alto.

## 0. Configuración

In [1]:
import unicodedata
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import statsmodels.api as sm
from scipy.stats import chi2_contingency
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (accuracy_score, cohen_kappa_score, confusion_matrix, f1_score,
                             make_scorer, mean_absolute_error, precision_score, r2_score, recall_score,
                             root_mean_squared_error)
from sklearn.model_selection import (GridSearchCV, GroupKFold, GroupShuffleSplit, StratifiedGroupKFold,
                                     cross_validate)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, SplineTransformer, StandardScaler

# Aviso esperado: una variedad o suelo que aparece en prueba y no en entrenamiento se manda al grupo "poco frecuente"
warnings.filterwarnings("ignore", message="Found unknown categories")

SEMILLA = 42  # una sola semilla para que todo el notebook sea reproducible
FIGURAS = Path.cwd() / "figuras"  # aquí se guardan las figuras que van al informe
FIGURAS.mkdir(exist_ok=True)
COLOR = {"azul": "#2a78d6", "naranja": "#eb6834", "verde": "#1baf7a", "gris": "#52514e"}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.25, "grid.linewidth": 0.5, "font.size": 10})
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
print(f"pandas {pd.__version__} | numpy {np.__version__} | scikit-learn {sklearn.__version__}")

pandas 3.0.6 | numpy 2.5.3 | scikit-learn 1.9.1


## 1. Datos, targets y partición 80/20

### 1.1 Carga de los datos

Los datos son confidenciales y no se incluyen en la entrega. El notebook los busca en una carpeta `Datos/`
(en esta carpeta o en las de arriba) y acepta CSV o Excel.

In [2]:
def buscar_datos(nombre):
    """Devuelve la ruta del archivo dentro de una carpeta Datos/. Si hay CSV lo prefiere; si no, usa el Excel."""
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        for extension in (".csv", ".xlsx"):
            ruta = carpeta / "Datos" / f"{nombre}{extension}"
            if ruta.exists():
                return ruta
    raise FileNotFoundError(f"No se encontró {nombre}.csv ni {nombre}.xlsx en una carpeta Datos/")


def leer(ruta, columna_indice=None):
    """Lee un CSV o la primera hoja de un Excel."""
    if ruta.suffix == ".xlsx":
        return pd.read_excel(ruta, sheet_name=0, index_col=columna_indice)
    return pd.read_csv(ruta, encoding="utf-8-sig", index_col=columna_indice, low_memory=False)


historico_crudo = leer(buscar_datos("HISTORICO_SUERTES"))
ipsa = leer(buscar_datos("BD_IPSA_1940"), columna_indice=0)  # la primera columna es un índice sin nombre
ipsa["TAL"] = ipsa["TAL"].astype(str).str.strip()
ipsa = ipsa.reset_index(drop=True)

print(f"HISTORICO_SUERTES: {historico_crudo.shape[0]:,} filas x {historico_crudo.shape[1]} columnas")
print(f"BD_IPSA_1940:      {ipsa.shape[0]:,} filas x {ipsa.shape[1]} columnas")

HISTORICO_SUERTES: 21,027 filas x 85 columnas
BD_IPSA_1940:      2,187 filas x 20 columnas


### 1.2 Filas que entran al modelo de regresión

Del taller anterior se mantienen dos filtros, que son reglas fijas (no usan estadísticos de los datos, así que
se pueden aplicar antes de partir):

1. Se quitan las cosechas destinadas a **semilla**: no son producción comercial.
2. Se quitan los TCH imposibles (< 50 o > 250) en suertes de **1 ha o menos**: en lotes tan pequeños un
   error pequeño en el área dispara el TCH.

In [3]:
historico = historico_crudo[historico_crudo["Destino 1=Semilla"] != 1]
n_sin_semilla = len(historico)

error_area = ((historico["TCH"] < 50) | (historico["TCH"] > 250)) & (historico["Area Neta"] <= 1)
historico = historico[~error_area].reset_index(drop=True)

print(f"Filas originales:               {len(historico_crudo):,}")
print(f"Sin cosechas de semilla:        {n_sin_semilla:,}  (-{len(historico_crudo) - n_sin_semilla})")
print(f"Sin TCH imposible en <= 1 ha:   {len(historico):,}  (-{n_sin_semilla - len(historico)})")

Filas originales:               21,027
Sin cosechas de semilla:        20,733  (-294)
Sin TCH imposible en <= 1 ha:   20,702  (-31)


### 1.3 Variables predictoras

La regla para escoger predictoras es una sola: **la variable se tiene que conocer antes de cosechar**. Por eso
quedan por fuera las mediciones de fábrica y laboratorio (Brix, pureza, %ATR, materia extraña), las toneladas
cosechadas (de ahí sale el TCH) y el tiempo entre el corte y el molino. Usarlas sería fuga de información: el
modelo acertaría con datos que no existen cuando se necesita la predicción.

In [4]:
# --- Regresión (HISTORICO_SUERTES) ---
LLUVIAS_VENTANA = ["Lluvias (2 Meses Ant.)", "Lluvias 0 -3", "Lluvias tres a seis", "Lluvias seis a nueve",
                   "Luvias 9 -FC"]  # así viene escrito en el archivo
PREDICTORAS_REG = ["Edad Ult Cos", "Cod.Estado #", "Variedad", "Suelo", "Zona", "Dist Km", "Producto",
                   "Dosis Madurante", "Semanas mad.", "T.Corte", "Tenencia", "Cultivo", *LLUVIAS_VENTANA,
                   "M3 Riego", "DDUlt.Riego", "Período"]
AUXILIARES_REG = ["Temp. Media Ciclo"]  # solo sirve para marcar si la cosecha tiene dato climático
TARGETS_REG = ["TCH", "%Sac.Caña"]

# --- Clasificación (BD_IPSA_1940) ---
PREDICTORAS_CLF = ["edad", "cortes", "semsmad", "dosismad", "lluvias", "grupo_tenencia", "mes", "periodo"]
TARGETS_CLF = ["TCH", "sacarosa"]

constantes = {c: ipsa[c].iloc[0] for c in ipsa.columns if ipsa[c].nunique() == 1}
print(f"Predictoras de regresión: {len(PREDICTORAS_REG)} | de clasificación: {len(PREDICTORAS_CLF)}")
print("Columnas constantes en BD_IPSA_1940 (no sirven como predictoras):")
for columna, valor in constantes.items():
    print(f"   {columna} = {valor}")

Predictoras de regresión: 20 | de clasificación: 8
Columnas constantes en BD_IPSA_1940 (no sirven como predictoras):
   tipocorte = Mecanizado Verde
   variedad = CC01-1940
   madurada = SI
   producto = BONUS 250 EC REGULADOR FISIOLÓGICO


BD_IPSA_1940 tiene una sola variedad (CC01-1940), madurada con el mismo producto y cosechada a máquina. Los
modelos de clasificación **solo aplican a esa población**.

En clasificación quedan por fuera `me` (materia extraña, se mide en fábrica), `vejez` (horas entre el corte y
el molino) y `pct_diatrea` (en el taller anterior se encontró que solo cerca del 25 % de sus valores son
mediciones reales).

### 1.4 Niveles Bajo, Medio y Alto

Las clases se definen con los umbrales de referencia del Ingenio. Son fijos y no se calculan con los datos, así
que no pasan información del conjunto de prueba al de entrenamiento.

| Target | Bajo | Medio | Alto |
|---|---|---|---|
| TCH (t/ha) | < 125 | 125 a 150 | > 150 |
| Sacarosa (%) | < 12.2 | 12.2 a 12.9 | ≥ 13.0 |

In [5]:
CLASES = ["Bajo", "Medio", "Alto"]
UMBRALES = {"TCH": (125, 151), "sacarosa": (12.2, 13.0)}  # [b1, b2): Medio va de b1 hasta antes de b2


def categorizar(y, target):
    """Convierte TCH o sacarosa en Bajo / Medio / Alto con los umbrales fijos."""
    b1, b2 = UMBRALES[target]
    # se redondea a 1 decimal para que un 12.2 guardado como 12.19999 no caiga en la clase equivocada
    return pd.cut(y.round(1), [-np.inf, b1, b2, np.inf], right=False, labels=CLASES)


reparto = pd.DataFrame({t: categorizar(ipsa[t], t).value_counts(normalize=True).reindex(CLASES) * 100
                        for t in TARGETS_CLF}).round(1)
print("Porcentaje de cosechas por clase:")
display(reparto.T)

Porcentaje de cosechas por clase:


,Bajo,Medio,Alto
TCH,22.8,38.9,38.4
sacarosa,21.5,36.7,41.8


La clase Bajo es la más pequeña (cerca del 22 %) y es la que más le interesa al Ingenio detectar. Por eso más
adelante se balancean las clases y se mira el recall de Bajo, además de las métricas generales.

### 1.5 Partición de entrenamiento (80 %) y prueba (20 %)

La partición se hace **antes de cualquier cálculo** y el conjunto de prueba no se vuelve a tocar hasta la
evaluación final. No se parte fila por fila sino **por grupos**:

- **Regresión: por suerte** (hacienda + suerte). Una misma suerte aparece varias veces (una por cosecha). Si
  unas cosechas quedan en entrenamiento y otras en prueba, el modelo ya "conoce" ese lote y el resultado sale
  mejor de lo que sería con un lote nuevo.
- **Clasificación: por hacienda**, y además estratificada por clase. En el taller anterior se vio que partir por
  lote dejaba lotes vecinos de la misma hacienda y el mismo mes a los dos lados, y KNN los memorizaba.

La validación cruzada (5 particiones) se hace **solo sobre entrenamiento** y respeta los mismos grupos.

In [6]:
def grupos_suerte(df):
    """Identificador del lote en HISTORICO_SUERTES: el código de suerte se repite entre haciendas."""
    return df["Hacienda"].astype(str) + "_" + df["Suerte"].astype(str).str.strip()


def grupos_hacienda(df):
    """Identificador de la hacienda en BD_IPSA_1940."""
    return df["FAZ"].astype(str)


# --- Regresión: 80/20 por suerte ---
particion = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEMILLA)
i_train, i_test = next(particion.split(historico, groups=grupos_suerte(historico)))
train_reg = historico.iloc[i_train].reset_index(drop=True)
test_reg = historico.iloc[i_test].reset_index(drop=True)

# --- Clasificación: 80/20 por hacienda, estratificado por la clase de TCH y de sacarosa a la vez ---
clase_doble = categorizar(ipsa["TCH"], "TCH").astype(str) + "_" + categorizar(ipsa["sacarosa"], "sacarosa").astype(str)
particion = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
i_train, i_test = next(particion.split(ipsa, clase_doble, groups=grupos_hacienda(ipsa)))  # 1 de 5 partes = 20 %
train_clf = ipsa.iloc[i_train].reset_index(drop=True)
test_clf = ipsa.iloc[i_test].reset_index(drop=True)

# --- Validación cruzada, solo para entrenamiento ---
CV_REG = GroupKFold(n_splits=5)
CV_CLF = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

resumen = pd.DataFrame({
    "filas train": [len(train_reg), len(train_clf)],
    "filas test": [len(test_reg), len(test_clf)],
    "% test": [100 * len(test_reg) / len(historico), 100 * len(test_clf) / len(ipsa)],
    "grupos train": [grupos_suerte(train_reg).nunique(), grupos_hacienda(train_clf).nunique()],
    "grupos test": [grupos_suerte(test_reg).nunique(), grupos_hacienda(test_clf).nunique()],
    "grupos en los dos lados": [len(set(grupos_suerte(train_reg)) & set(grupos_suerte(test_reg))),
                                len(set(grupos_hacienda(train_clf)) & set(grupos_hacienda(test_clf)))],
}, index=["Regresión (grupo = suerte)", "Clasificación (grupo = hacienda)"]).round(1)
display(resumen)

assert (resumen["grupos en los dos lados"] == 0).all(), "hay grupos repetidos entre train y test"
assert resumen["% test"].between(19, 21).all(), "la prueba no quedó cerca del 20 %"

,filas train,filas test,% test,grupos train,grupos test,grupos en los dos lados
Regresión (grupo = suerte),16598,4104,19.8,4608,1152,0
Clasificación (grupo = hacienda),1749,438,20.0,229,56,0


### 1.6 ¿Entrenamiento y prueba se parecen?

Si la partición quedó bien, los targets deben tener una distribución parecida en los dos conjuntos.

In [7]:
comparacion_reg = pd.DataFrame({
    (t, nombre): df[t].describe()[["mean", "25%", "50%", "75%"]]
    for t in TARGETS_REG for nombre, df in (("train", train_reg), ("test", test_reg))
}).T.round(2)
print("Regresión: targets en entrenamiento y en prueba")
display(comparacion_reg)

filas = {}
for t in TARGETS_CLF:
    c_train, c_test = categorizar(train_clf[t], t), categorizar(test_clf[t], t)
    tabla = pd.DataFrame({"train": c_train.value_counts().reindex(CLASES), "test": c_test.value_counts().reindex(CLASES)})
    p = chi2_contingency(tabla)[1]
    filas[(t, "train")] = (tabla["train"] / tabla["train"].sum() * 100).round(1).to_dict() | {"p (chi2)": round(p, 3)}
    filas[(t, "test")] = (tabla["test"] / tabla["test"].sum() * 100).round(1).to_dict() | {"p (chi2)": round(p, 3)}
print("Clasificación: % de cada clase en entrenamiento y en prueba")
display(pd.DataFrame(filas).T)

Regresión: targets en entrenamiento y en prueba


mean     25%     50%     75%
TCH       train  130.22  109.06  129.92  151.04
          test   129.79  110.60  129.44  149.04
%Sac.Caña train   12.32   11.61   12.36   13.09
          test    12.34   11.67   12.40   13.09

Clasificación: % de cada clase en entrenamiento y en prueba


Bajo  Medio  Alto  p (chi2)
TCH      train  22.7   38.9  38.4     0.986
         test   23.1   38.8  38.1     0.986
sacarosa train  21.6   36.7  41.7     0.995
         test   21.5   36.5  42.0     0.995

Las medias y los cuartiles de TCH y %Sac.Caña son casi iguales en entrenamiento y prueba, y en clasificación
el reparto de clases es prácticamente el mismo (p alto en la prueba chi-cuadrado: no hay diferencia entre los
dos conjuntos). La partición es válida.

### 1.7 Funciones para armar X, y y los grupos

Los modelos reciben las columnas crudas; todo el preprocesamiento ocurre dentro del `Pipeline` (sección 2).

In [8]:
def preparar_reg(df, target):
    """X, y y grupos (suerte) para regresión. Las filas sin target se descartan: el target no se imputa."""
    d = df[df[target].notna()].reset_index(drop=True)
    return d[PREDICTORAS_REG + AUXILIARES_REG], d[target], grupos_suerte(d)


def preparar_clf(df, target):
    """X, clase (Bajo / Medio / Alto) y grupos (hacienda) para clasificación."""
    return df[PREDICTORAS_CLF], categorizar(df[target], target).astype(str), grupos_hacienda(df)


for t in TARGETS_REG:
    X, y, g = preparar_reg(train_reg, t)
    print(f"Regresión {t}: X {X.shape}, {g.nunique():,} suertes, {len(train_reg) - len(X)} filas de train sin target")
for t in TARGETS_CLF:
    X, y, g = preparar_clf(train_clf, t)
    print(f"Clasificación {t}: X {X.shape}, {g.nunique()} haciendas")

Regresión TCH: X (16598, 21), 4,608 suertes, 0 filas de train sin target
Regresión %Sac.Caña: X (16471, 21), 4,598 suertes, 127 filas de train sin target
Clasificación TCH: X (1749, 8), 229 haciendas
Clasificación sacarosa: X (1749, 8), 229 haciendas


## 2. Preprocesamiento dentro de un `Pipeline`

El preprocesamiento va **dentro del `Pipeline` de cada modelo**. Así, en la validación cruzada se vuelve a
ajustar con el entrenamiento de cada partición y nunca ve los datos con los que se evalúa. Tiene dos pasos:

1. **Reglas de negocio.** Son reglas fijas, fila por fila. No aprenden nada de los datos (por ejemplo, "una
   lluvia de 0 mm es un dato faltante").
2. **Transformaciones que sí aprenden** (medianas para imputar, categorías, medias y desviaciones para escalar).
   Estas se calculan **solo con entrenamiento**.

Hay dos versiones del paso 2, según el tipo de modelo:

| | Modelos lineales, logística y KNN | Modelos de árboles (Random Forest, XGBoost) |
|---|---|---|
| Numéricas | Estandarizadas (media 0, desviación 1) | En su escala original |
| Lluvias | `log1p` (tienen cola larga) + estandarizar | En su escala original |
| Semanas de madurante | Spline (su efecto sube y luego baja) | Tal cual: el árbol encuentra los cortes solo |
| Categóricas | One-hot quitando una categoría (evita colinealidad) | One-hot con todas las categorías |

Los árboles no necesitan escalado ni transformaciones porque solo comparan si un valor es mayor o menor que un
punto de corte.

### 2.1 Reglas de negocio de regresión (HISTORICO_SUERTES)

Vienen del análisis exploratorio del taller anterior:

| Caso | Qué se hace | Por qué |
|---|---|---|
| Lluvia = 0 en una ventana de meses | Se pasa a dato faltante y se crea un indicador | En el Valle del Cauca no pasan tres meses sin llover: el 0 es "no se registró" (sobre todo en proveedores) |
| Lluvia por encima de 300 mm/mes | Se recorta a ese tope | Límite físico |
| Madurante sin producto | Producto = "NO APLICADO", dosis y semanas = 0 | No se aplicó madurante |
| Producto informado con dosis 0 | Dosis faltante | Error de captura |
| Semanas de madurante negativas o más largas que el ciclo | Faltante | Error de fecha |
| `M3 Riego` = 0 | Se deja el 0 y se crea un indicador de registro de riego | No se distingue "no se regó" de "no se registró" |
| Edad mayor a 36 meses | Se recorta a 36 | Caso extremo aislado |
| `Período` (año-mes) | Se separa en año y mes de cosecha | Tendencia y época del año |
| Temperatura media del ciclo | Solo se usa para un indicador de "hay dato climático" | Está vacía en la mayoría de cosechas |

In [9]:
LLUVIA_2M, LLUVIA_FC = "Lluvias (2 Meses Ant.)", "Luvias 9 -FC"
LLUVIAS_TRIMESTRALES = ["Lluvias 0 -3", "Lluvias tres a seis", "Lluvias seis a nueve"]
TOPE_MM_MES = 300            # límite físico de lluvia
SEMANAS_POR_MES = 30.44 / 7
TOPE_EDAD_MESES = 36


def normalizar_texto(s):
    """Quita acentos, pasa a mayúsculas y deja un solo espacio (para que FISIOLÓGICO = FISIOLOGICO)."""
    def limpiar(v):
        v = unicodedata.normalize("NFKD", v).encode("ascii", "ignore").decode()
        return " ".join(v.upper().split())
    return s.map(limpiar, na_action="ignore")


class ReglasNegocio(TransformerMixin, BaseEstimator):
    """Reglas fijas de regresión. No aprende nada en fit: transforma cada fila por separado."""

    def fit(self, X, y=None):
        self.columnas_salida_ = self._aplicar(X.head(1)).columns.to_numpy(dtype=object)
        return self

    def transform(self, X):
        return self._aplicar(X)

    def get_feature_names_out(self, input_features=None):
        return self.columnas_salida_

    @staticmethod
    def _aplicar(X):
        X = X.copy()
        edad = X["Edad Ult Cos"]

        # Año y mes de cosecha; indicador de dato climático
        X["anio"] = X["Período"] // 100
        X["mes_cosecha"] = X["Período"] % 100
        X["tiene_dato_climatico"] = X["Temp. Media Ciclo"].notna().astype(int)
        X = X.drop(columns=["Período", "Temp. Media Ciclo"])

        X["Zona"] = X["Zona"].str.upper()
        X["Suelo"] = X["Suelo"].str.strip().fillna("Desconocido")
        X["Tenencia"] = X["Tenencia"].map(lambda v: str(int(v)), na_action="ignore")  # es un código, no una cantidad

        # Madurante: vacío = no aplicado; con dosis pero sin nombre = desconocido
        producto = normalizar_texto(X["Producto"])
        dosis = X["Dosis Madurante"]
        producto = producto.mask(producto.isna() & (dosis > 0), "DESCONOCIDO").fillna("NO APLICADO")
        madurada = producto != "NO APLICADO"
        X["Producto"] = producto
        X["madurada"] = madurada.astype(int)
        X["Dosis Madurante"] = dosis.mask(madurada & (dosis == 0)).where(madurada, 0.0)  # dosis 0 con producto: error
        semanas = X["Semanas mad."]
        fecha_imposible = (semanas < 0) | (semanas > edad * SEMANAS_POR_MES)
        X["Semanas mad."] = semanas.mask(fecha_imposible).where(madurada, 0.0)

        # Lluvias: 0 = sin dato; recorte al tope físico según el largo de cada ventana
        lluvias = X[LLUVIAS_VENTANA].astype(float).replace(0, np.nan)
        lluvias[LLUVIAS_TRIMESTRALES] = lluvias[LLUVIAS_TRIMESTRALES].clip(upper=3 * TOPE_MM_MES)
        lluvias[LLUVIA_2M] = lluvias[LLUVIA_2M].clip(upper=2 * TOPE_MM_MES)
        meses_fc = edad - 9  # la última ventana va del mes 9 hasta el corte
        lluvias[LLUVIA_FC] = lluvias[LLUVIA_FC].clip(upper=(TOPE_MM_MES * meses_fc).where(meses_fc >= 1))
        lluvias.loc[meses_fc <= 0, LLUVIA_FC] = 0.0  # si se cortó antes del mes 9, esa ventana no existe: 0 real
        X[LLUVIAS_VENTANA] = lluvias

        # Riego
        X["tiene_registro_riego"] = (X["M3 Riego"] > 0).astype(int)
        X["DDUlt.Riego"] = X["DDUlt.Riego"].astype(float).mask(X["M3 Riego"] == 0)  # días desde un riego que no existe

        X["Edad Ult Cos"] = edad.clip(upper=TOPE_EDAD_MESES)
        return X


# Cuántas cosechas toca cada regla (son reglas fijas, así que se pueden contar sobre todos los datos)
X_todo = historico[PREDICTORAS_REG + AUXILIARES_REG]
R = ReglasNegocio().fit_transform(X_todo)
lluvia_original = X_todo[LLUVIAS_VENTANA].astype(float)
conteo = pd.Series({
    "Lluvia = 0 en alguna ventana -> sin dato": (lluvia_original == 0).any(axis=1).sum(),
    "Lluvia recortada al tope físico": ((R[LLUVIAS_VENTANA] < lluvia_original) & (R[LLUVIAS_VENTANA] > 0)).any(axis=1).sum(),
    "Madurante no aplicado": (R["madurada"] == 0).sum(),
    "Producto con dosis 0 -> sin dato": (R["Dosis Madurante"].isna() & (X_todo["Dosis Madurante"] == 0)).sum(),
    "Semanas de madurante imposibles -> sin dato": (R["Semanas mad."].isna() & X_todo["Semanas mad."].notna()).sum(),
    "Sin registro de riego": (R["tiene_registro_riego"] == 0).sum(),
    "Sin dato climático": (R["tiene_dato_climatico"] == 0).sum(),
    "Edad recortada a 36 meses": (X_todo["Edad Ult Cos"] > TOPE_EDAD_MESES).sum(),
})
display(pd.DataFrame({"cosechas": conteo, "% del total": (100 * conteo / len(X_todo)).round(1)}))

,cosechas,% del total
Lluvia = 0 en alguna ventana -> sin dato,11518,55.6
Lluvia recortada al tope físico,36,0.2
Madurante no aplicado,9468,45.7
Producto con dosis 0 -> sin dato,14,0.1
Semanas de madurante imposibles -> sin dato,10,0.0
Sin registro de riego,12501,60.4
Sin dato climático,16283,78.7
Edad recortada a 36 meses,1,0.0


Lo que más pesa es la falta de registro: lluvia, riego y clima faltan en una parte grande de las cosechas. Por
eso se imputa y, además, se le dice al modelo dónde faltaba el dato (con los indicadores), porque la falta de
registro no es al azar: se concentra en los lotes de proveedores.

### 2.2 Preprocesador de regresión

Para imputar se usa la **mediana de un grupo parecido** en vez de una sola mediana global: la lluvia faltante
se llena con la mediana de la misma zona y el mismo mes, y la dosis con la mediana del mismo producto. Si el
grupo no existe en entrenamiento, se usa un grupo más amplio y, al final, la mediana general.

In [10]:
class ImputadorMedianaGrupo(TransformerMixin, BaseEstimator):
    """Imputa con la mediana de entrenamiento por grupo.

    `niveles` va del grupo más fino al más amplio; el último respaldo es la mediana general.
    Con `filtro`, las medianas se calculan solo con las filas donde esa columna vale 1 (por ejemplo, solo
    las cosechas maduradas, para que los ceros de "no aplicado" no bajen la mediana).
    """

    def __init__(self, columnas, niveles=(), filtro=None):
        self.columnas = columnas
        self.niveles = niveles
        self.filtro = filtro

    def fit(self, X, y=None):
        cols = list(self.columnas)
        base = X[X[self.filtro] == 1] if self.filtro else X
        self.medianas_ = [base.groupby(list(nivel))[cols].median() for nivel in self.niveles]
        self.mediana_global_ = base[cols].median()
        return self

    def transform(self, X):
        salida = X[list(self.columnas)].copy()
        for nivel, medianas in zip(self.niveles, self.medianas_):
            nivel = list(nivel)
            clave = pd.MultiIndex.from_frame(X[nivel]) if len(nivel) > 1 else pd.Index(X[nivel[0]])
            salida = salida.fillna(medianas.reindex(clave).set_axis(X.index))
        return salida.fillna(self.mediana_global_)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.columnas, dtype=object)


NUM_REG = ["Edad Ult Cos", "Cod.Estado #", "Dist Km", "anio", "M3 Riego"]
CAT_REG = ["Variedad", "Suelo", "Zona", "Producto", "T.Corte", "Tenencia", "Cultivo", "mes_cosecha"]
FLAGS_REG = ["madurada", "tiene_registro_riego", "tiene_dato_climatico"]
# Nodos fijos del spline de semanas de madurante (no dependen de los datos). Después de 40 semanas hay muy
# pocos datos y el efecto se deja constante.
NODOS_SEMANAS = np.array([0, 6, 12, 18, 30, 40], dtype=float).reshape(-1, 1)


def preprocesador_reg(tipo="lineal"):
    """Preprocesamiento de regresión. tipo="lineal": log1p, spline y escalado. tipo="arbol": escala original."""
    lineal = tipo == "lineal"
    escalar = [StandardScaler()] if lineal else []
    log1p = [FunctionTransformer(np.log1p, feature_names_out="one-to-one")] if lineal else []

    imp_lluvia = ImputadorMedianaGrupo(LLUVIAS_VENTANA, niveles=(("Zona", "mes_cosecha"), ("Zona",)))
    imp_dosis = ImputadorMedianaGrupo(["Dosis Madurante"], niveles=(("Producto",),), filtro="madurada")
    imp_semanas = ImputadorMedianaGrupo(["Semanas mad."], filtro="madurada")
    if lineal:
        # sin include_bias: las bases del spline suman 1 y repetirían el intercepto
        semanas = make_pipeline(imp_semanas, SplineTransformer(knots=NODOS_SEMANAS, extrapolation="constant",
                                                               include_bias=False))
    else:
        semanas = imp_semanas
    # En el lineal, "madurada" ya queda dicha por las columnas de Producto (madurada = 0 es "NO APLICADO")
    flags = [f for f in FLAGS_REG if f != "madurada"] if lineal else FLAGS_REG

    bloques = [
        ("lluvias", make_pipeline(imp_lluvia, *log1p, *escalar), LLUVIAS_VENTANA + ["Zona", "mes_cosecha"]),
        ("sin_dato_lluvia", MissingIndicator(features="missing-only"), LLUVIAS_VENTANA),
        ("dosis", make_pipeline(imp_dosis, *escalar), ["Dosis Madurante", "Producto", "madurada"]),
        ("semanas", semanas, ["Semanas mad.", "madurada"]),
        ("dd_riego", make_pipeline(SimpleImputer(strategy="median"), *escalar), ["DDUlt.Riego"]),
        ("numericas", make_pipeline(SimpleImputer(strategy="median"), *escalar), NUM_REG),
        # Las categorías con menos del 1 % de las cosechas se juntan en una sola ("poco frecuente")
        ("categoricas", make_pipeline(
            SimpleImputer(strategy="most_frequent"),
            OneHotEncoder(min_frequency=0.01, handle_unknown="infrequent_if_exist",
                          drop="first" if lineal else None, sparse_output=False)), CAT_REG),
        ("flags", "passthrough", flags),
    ]
    pasos = [("reglas", ReglasNegocio()),
             ("columnas", ColumnTransformer(bloques, remainder="drop", verbose_feature_names_out=False))]
    if lineal:
        pasos.append(("sin_constantes", VarianceThreshold()))  # quita columnas que quedan con un solo valor
    return Pipeline(pasos).set_output(transform="pandas")

### 2.3 Reglas y preprocesador de clasificación (BD_IPSA_1940)

Las reglas siguen la misma lógica que en regresión, adaptadas a esta base:

| Caso | Qué se hace | Por qué |
|---|---|---|
| `lluvias` = 0 | Faltante + indicador `falta_dato_lluvia` | Igual que en regresión: es falta de registro |
| `dosismad` = 0 o mayor a 3 | Faltante | Todas las cosechas están maduradas y las dosis reales van de 0.2 a 2.0 |
| `semsmad` negativa | Faltante | El madurante figura aplicado después del corte |
| `mes`, `periodo` | Mes y año de cosecha | Época del año y tendencia |

Hay tres versiones del preprocesador:

- **`logistica`**: todas las predictoras, escaladas, con spline en las semanas de madurante.
- **`knn`**: KNN mide distancias y no sabe ignorar variables, así que se le quitan las que en el taller anterior
  resultaron poco informativas para cada target.
- **`arbol`** (nueva en este caso): todas las predictoras en su escala original, para Random Forest y XGBoost.

In [11]:
TOPE_DOSIS = 3
INICIO_VENTANA_LLUVIA = 10   # la lluvia de BD_IPSA corresponde, aproximadamente, a la del mes 10 hasta el corte
NUM_CLF = ["edad", "cortes", "anio"]
CAT_CLF = ["grupo_tenencia", "mes_cosecha"]
FLAG_LLUVIA = "falta_dato_lluvia"
# Variables que KNN no usa en cada target (poco informativas según el análisis del taller anterior)
EXCLUIDAS_KNN = {"TCH": {"semsmad", "mes_cosecha", "dosismad"},
                 "sacarosa": {"cortes", "grupo_tenencia", FLAG_LLUVIA}}


class ReglasNegocioClasificacion(TransformerMixin, BaseEstimator):
    """Reglas fijas de clasificación. No aprende nada en fit."""

    def fit(self, X, y=None):
        self.columnas_salida_ = self._aplicar(X.head(1)).columns.to_numpy(dtype=object)
        return self

    def transform(self, X):
        return self._aplicar(X)

    def get_feature_names_out(self, input_features=None):
        return self.columnas_salida_

    @staticmethod
    def _aplicar(X):
        X = X.copy()
        X["anio"] = X["periodo"] // 100
        X["mes_cosecha"] = X["mes"]
        X = X.drop(columns=["mes", "periodo"])

        dosis = X["dosismad"].astype(float)
        X["dosismad"] = dosis.mask((dosis == 0) | (dosis > TOPE_DOSIS))
        semanas = X["semsmad"].astype(float)
        X["semsmad"] = semanas.mask(semanas < 0)

        ventana = X["edad"] - INICIO_VENTANA_LLUVIA
        lluvias = X["lluvias"].astype(float).replace(0, np.nan)
        X["lluvias"] = lluvias.clip(upper=(TOPE_MM_MES * ventana).where(ventana >= 1))
        return X


def falta_dato(X):
    """1 cuando falta el dato de lluvia."""
    return X.isna().astype(int).set_axis([FLAG_LLUVIA], axis=1)


def nombre_flag_lluvia(transformador, columnas):
    return np.array([FLAG_LLUVIA], dtype=object)


def preprocesador_clf(tipo="logistica", target=None):
    """Preprocesamiento de clasificación: tipo = "logistica", "knn" (necesita el target) o "arbol"."""
    if tipo == "knn" and target not in EXCLUIDAS_KNN:
        raise ValueError("KNN necesita target='TCH' o 'sacarosa' para saber qué variables usar")
    excluidas = EXCLUIDAS_KNN[target] if tipo == "knn" else set()
    usar = lambda c: c not in excluidas
    arbol = tipo == "arbol"
    escalar = [] if arbol else [StandardScaler()]
    log1p = [] if arbol else [FunctionTransformer(np.log1p, feature_names_out="one-to-one")]

    # lluvia faltante: mediana del mismo grupo de tenencia y mes; si no hay, la del mes
    imp_lluvia = ImputadorMedianaGrupo(["lluvias"], niveles=(("grupo_tenencia", "mes_cosecha"), ("mes_cosecha",)))
    if tipo == "logistica":
        semanas = make_pipeline(SimpleImputer(strategy="median"),
                                SplineTransformer(knots=NODOS_SEMANAS, extrapolation="constant", include_bias=False))
    else:
        semanas = make_pipeline(SimpleImputer(strategy="median"), *escalar)

    bloques = [
        ("lluvias", make_pipeline(imp_lluvia, *log1p, *escalar), ["lluvias", "grupo_tenencia", "mes_cosecha"]),
        ("numericas", make_pipeline(SimpleImputer(strategy="median"), *escalar), [c for c in NUM_CLF if usar(c)]),
        ("categoricas", OneHotEncoder(handle_unknown="ignore", drop="first" if tipo == "logistica" else None,
                                      sparse_output=False), [c for c in CAT_CLF if usar(c)]),
    ]
    if usar(FLAG_LLUVIA):
        bloques.append((FLAG_LLUVIA, FunctionTransformer(falta_dato, feature_names_out=nombre_flag_lluvia), ["lluvias"]))
    if usar("dosismad"):
        bloques.append(("dosis", make_pipeline(SimpleImputer(strategy="median"), *escalar), ["dosismad"]))
    if usar("semsmad"):
        bloques.append(("semanas", semanas, ["semsmad"]))
    return Pipeline([("reglas", ReglasNegocioClasificacion()),
                     ("columnas", ColumnTransformer(bloques, remainder="drop", verbose_feature_names_out=False))]
                    ).set_output(transform="pandas")


X_ipsa = ipsa[PREDICTORAS_CLF]
R = ReglasNegocioClasificacion().fit_transform(X_ipsa)
conteo = pd.Series({
    "lluvias = 0 -> sin dato": R["lluvias"].isna().sum(),
    "dosismad = 0 o > 3 -> sin dato": R["dosismad"].isna().sum(),
    "semsmad negativa -> sin dato": R["semsmad"].isna().sum(),
})
display(pd.DataFrame({"cosechas": conteo, "% del total": (100 * conteo / len(X_ipsa)).round(1)}))

,cosechas,% del total
lluvias = 0 -> sin dato,685,31.3
dosismad = 0 o > 3 -> sin dato,2,0.1
semsmad negativa -> sin dato,2,0.1


### 2.4 Comprobaciones

Antes de entrenar se revisa que cada versión (a) se ajusta con entrenamiento y transforma la prueba sin dejar
datos faltantes, y (b) aprende sus valores **solo de entrenamiento**.

In [12]:
filas = []
for t in TARGETS_REG:
    X_tr, _, _ = preparar_reg(train_reg, t)
    X_te, _, _ = preparar_reg(test_reg, t)
    for tipo in ("lineal", "arbol"):
        pre = preprocesador_reg(tipo).fit(X_tr)          # se ajusta solo con entrenamiento
        Z_tr, Z_te = pre.transform(X_tr), pre.transform(X_te)
        filas.append({"problema": f"Regresión {t}", "versión": tipo, "filas train": len(Z_tr), "filas test": len(Z_te),
                      "columnas": Z_tr.shape[1], "faltantes train": int(Z_tr.isna().sum().sum()),
                      "faltantes test": int(Z_te.isna().sum().sum())})

X_tr, X_te = train_clf[PREDICTORAS_CLF], test_clf[PREDICTORAS_CLF]
versiones_clf = {"logistica": {}, "knn (TCH)": {"target": "TCH"}, "knn (sacarosa)": {"target": "sacarosa"}, "arbol": {}}
columnas_clf = {}
for nombre, extra in versiones_clf.items():
    pre = preprocesador_clf(nombre.split(" ")[0], **extra).fit(X_tr)
    Z_tr, Z_te = pre.transform(X_tr), pre.transform(X_te)
    columnas_clf[nombre] = list(Z_tr.columns)
    filas.append({"problema": "Clasificación", "versión": nombre, "filas train": len(Z_tr), "filas test": len(Z_te),
                  "columnas": Z_tr.shape[1], "faltantes train": int(Z_tr.isna().sum().sum()),
                  "faltantes test": int(Z_te.isna().sum().sum())})

revision = pd.DataFrame(filas).set_index(["problema", "versión"])
display(revision)
assert (revision[["faltantes train", "faltantes test"]] == 0).all().all(), "quedaron datos faltantes"

print("Columnas de la versión para árboles (clasificación):")
print("  ", columnas_clf["arbol"])

filas train  filas test  columnas  faltantes train  faltantes test
problema            versión                                                                           
Regresión TCH       lineal                16598        4104        82                0               0
                    arbol                 16598        4104        85                0               0
Regresión %Sac.Caña lineal                16471        4075        82                0               0
                    arbol                 16471        4075        85                0               0
Clasificación       logistica              1749         438        26                0               0
                    knn (TCH)              1749         438         8                0               0
                    knn (sacarosa)         1749         438        17                0               0
                    arbol                  1749         438        22                0               0

Columnas de la versión para árboles (clasificación):
   ['lluvias', 'edad', 'cortes', 'anio', 'grupo_tenencia_1', 'grupo_tenencia_2', 'grupo_tenencia_3', 'mes_cosecha_1', 'mes_cosecha_2', 'mes_cosecha_3', 'mes_cosecha_4', 'mes_cosecha_5', 'mes_cosecha_6', 'mes_cosecha_7', 'mes_cosecha_8', 'mes_cosecha_9', 'mes_cosecha_10', 'mes_cosecha_11', 'mes_cosecha_12', 'falta_dato_lluvia', 'dosismad', 'semsmad']


In [13]:
# (b) Lo que aprende el preprocesador sale solo de entrenamiento
X_tr, _, _ = preparar_reg(train_reg, "TCH")
pre = preprocesador_reg("lineal").fit(X_tr)
escalador = pre.named_steps["columnas"].named_transformers_["numericas"].named_steps["standardscaler"]
edad_train = train_reg["Edad Ult Cos"].clip(upper=TOPE_EDAD_MESES).mean()
edad_todo = historico["Edad Ult Cos"].clip(upper=TOPE_EDAD_MESES).mean()
print(f"Media de la edad que guardó el escalador:   {escalador.mean_[0]:.4f}")
print(f"Media de la edad en entrenamiento:          {edad_train:.4f}")
print(f"Media de la edad en todos los datos:        {edad_todo:.4f}  (no es la que usa)")
assert np.isclose(escalador.mean_[0], edad_train)

imputador = pre.named_steps["columnas"].named_transformers_["lluvias"].named_steps["imputadormedianagrupo"]
R_tr = ReglasNegocio().fit_transform(X_tr)
esperadas = R_tr.groupby(["Zona", "mes_cosecha"])[LLUVIAS_VENTANA].median()
assert np.allclose(imputador.medianas_[0], esperadas, equal_nan=True)
print("Las medianas de lluvia por zona y mes coinciden con las calculadas solo en entrenamiento.")

Media de la edad que guardó el escalador:   13.0766
Media de la edad en entrenamiento:          13.0766
Media de la edad en todos los datos:        13.0638  (no es la que usa)
Las medianas de lluvia por zona y mes coinciden con las calculadas solo en entrenamiento.


**Resumen de la sección.** Quedan listos los preprocesadores que usarán todos los modelos: `preprocesador_reg`
(versiones `lineal` y `arbol`) y `preprocesador_clf` (versiones `logistica`, `knn` y `arbol`). Ninguna deja datos
faltantes y todas aprenden únicamente del conjunto de entrenamiento. Como van dentro del `Pipeline`, en la
validación cruzada se reajustan en cada partición.

## 3. Modelos de referencia: regresión lineal múltiple

La regresión lineal es el punto de partida: es simple, rápida y se puede leer coeficiente por coeficiente.
Los modelos avanzados de la sección 5 tienen que superarla para justificar su complejidad.

### 3.1 Cómo se evalúa cada modelo de regresión

Todos los modelos de regresión del notebook pasan por la misma función, para que la comparación sea justa:

1. **Validación cruzada de 5 particiones por suerte, solo con entrenamiento.** Da la media y la desviación de
   cada métrica.
2. **Ajuste con todo el entrenamiento y evaluación en la prueba**, que no participó en nada.

Métricas: **R²** (qué parte de la variación explica el modelo), **RMSE** (error típico, castiga más los errores
grandes) y **MAE** (error promedio en valor absoluto). RMSE y MAE quedan en las unidades del target.

In [14]:
METRICAS_REG = {"R2": "r2", "RMSE": "neg_root_mean_squared_error", "MAE": "neg_mean_absolute_error"}
resultados_reg = {}  # (target, modelo) -> métricas. Se llena aquí y en la sección 5; se compara en la sección 7
ajustados_reg = {}   # (target, modelo) -> modelo ajustado y sus predicciones en prueba


def evaluar_reg(nombre, target, modelo):
    """Validación cruzada por suerte en entrenamiento + ajuste con todo el entrenamiento + métricas en prueba."""
    X_tr, y_tr, g_tr = preparar_reg(train_reg, target)
    X_te, y_te, _ = preparar_reg(test_reg, target)

    cv = cross_validate(modelo, X_tr, y_tr, groups=g_tr, cv=CV_REG, scoring=list(METRICAS_REG.values()))
    fila = {}
    for metrica, clave in METRICAS_REG.items():
        valores = cv[f"test_{clave}"] * (-1 if clave.startswith("neg_") else 1)  # sklearn devuelve los errores en negativo
        fila[f"{metrica} CV"], fila[f"{metrica} CV (sd)"] = valores.mean(), valores.std()

    ajustado = clone(modelo).fit(X_tr, y_tr)
    pred = ajustado.predict(X_te)
    fila["R2 prueba"] = r2_score(y_te, pred)
    fila["RMSE prueba"] = root_mean_squared_error(y_te, pred)
    fila["MAE prueba"] = mean_absolute_error(y_te, pred)

    resultados_reg[(target, nombre)] = fila
    ajustados_reg[(target, nombre)] = {"modelo": ajustado, "y_prueba": y_te, "pred_prueba": pred}
    return fila


def tabla_reg():
    """Tabla con todos los modelos de regresión evaluados hasta el momento."""
    columnas = ["R2 CV", "R2 CV (sd)", "R2 prueba", "RMSE CV", "RMSE prueba", "MAE CV", "MAE prueba"]
    orden = [clave for t in TARGETS_REG for clave in resultados_reg if clave[0] == t]  # agrupada por target
    tabla = pd.DataFrame(resultados_reg).T.loc[orden, columnas]
    tabla.index.names = ["target", "modelo"]
    return tabla

### 3.2 Ajuste y desempeño

In [15]:
def modelo_lineal():
    return Pipeline([("pre", preprocesador_reg("lineal")), ("modelo", LinearRegression())])


for t in TARGETS_REG:
    evaluar_reg("Regresión lineal", t, modelo_lineal())
display(tabla_reg().round(3))

,,R2 CV,R2 CV (sd),R2 prueba,RMSE CV,RMSE prueba,MAE CV,MAE prueba
target,modelo,,,,,,,
TCH,Regresión lineal,0.328,0.015,0.314,26.581,26.096,20.409,20.008
%Sac.Caña,Regresión lineal,0.387,0.013,0.370,0.898,0.906,0.691,0.689


**Lectura.**

- **TCH:** el modelo explica cerca de un tercio de la variación (R² de 0.328 en validación cruzada y 0.314 en
  prueba). El error típico es de unas 26 t/ha (RMSE) y el error promedio de unas 20 t/ha (MAE).
- **%Sac.Caña:** explica un poco más (R² de 0.387 y 0.370), con un error típico de 0.9 puntos de sacarosa.
- **No hay sobreajuste:** las métricas de validación cruzada y de prueba son casi iguales, y la desviación entre
  particiones es pequeña (0.015 y 0.013 de R²). El modelo no falla por memorizar; falla porque es demasiado
  simple para estos datos.

### 3.3 Coeficientes: significancia, signo y magnitud

`LinearRegression` no entrega p-valores, así que el mismo modelo se ajusta con `statsmodels` (los coeficientes
son idénticos, se comprueba abajo). Los errores estándar se calculan **agrupados por suerte**: las cosechas de
una misma suerte se parecen entre sí, y si se trataran como independientes los p-valores saldrían demasiado
optimistas.

Cómo leer la magnitud:

- **Numéricas** (estandarizadas): cambio en el target cuando la variable sube **una desviación estándar**.
- **Categóricas**: diferencia frente a la categoría de referencia (la primera en orden alfabético).
- **Indicadores**: diferencia entre tener y no tener el dato.
- **Semanas de madurante**: va con spline, así que no se lee columna por columna sino como una curva (3.4).

In [16]:
def tipo_columna(c):
    if c.startswith("Semanas mad._sp"):
        return "spline"
    if c.startswith("missingindicator_") or c in FLAGS_REG:
        return "indicador"
    if c.startswith(tuple(f"{categorica}_" for categorica in CAT_REG)):
        return "categórica"
    return "numérica"


ols, coefs = {}, {}
for t in TARGETS_REG:
    X_tr, y_tr, g_tr = preparar_reg(train_reg, t)
    Z = preprocesador_reg("lineal").fit_transform(X_tr).astype(float)
    ols[t] = sm.OLS(y_tr.to_numpy(), sm.add_constant(Z, has_constant="add")).fit(
        cov_type="cluster", cov_kwds={"groups": pd.factorize(g_tr)[0]})  # errores agrupados por suerte
    ic = ols[t].conf_int()
    tabla = pd.DataFrame({"coef": ols[t].params, "error est.": ols[t].bse, "p-valor": ols[t].pvalues,
                          "IC 95% inf": ic[0], "IC 95% sup": ic[1]}).drop(index="const")
    tabla.insert(0, "tipo", tabla.index.map(tipo_columna))
    coefs[t] = tabla

    # mismos coeficientes que el LinearRegression de la sección 3.2
    lineal = ajustados_reg[(t, "Regresión lineal")]["modelo"].named_steps["modelo"]
    assert np.allclose(ols[t].params.drop("const"), lineal.coef_, rtol=1e-6, atol=1e-8)

resumen = pd.DataFrame({t: {"R2 en entrenamiento": round(ols[t].rsquared, 3), "cosechas": int(ols[t].nobs),
                            "coeficientes": len(coefs[t]),
                            "significativos (p < 0.05)": int((coefs[t]["p-valor"] < 0.05).sum())}
                        for t in TARGETS_REG})
display(resumen)

,TCH,%Sac.Caña
R2 en entrenamiento,0.341,0.395
cosechas,16598.000,16471.000
coeficientes,82.000,82.000
significativos (p < 0.05),47.000,36.000


In [17]:
# Variables numéricas: todas, también las no significativas
numericas = pd.concat({t: coefs[t].loc[coefs[t]["tipo"] == "numérica", ["coef", "p-valor"]] for t in TARGETS_REG}, axis=1)

# Cuánto vale "una desviación estándar" de cada variable (la que usó el escalador, en entrenamiento)
bloques = ajustados_reg[("TCH", "Regresión lineal")]["modelo"].named_steps["pre"].named_steps["columnas"].named_transformers_
una_sd = dict(zip(NUM_REG, bloques["numericas"].named_steps["standardscaler"].scale_))
una_sd["Dosis Madurante"] = bloques["dosis"].named_steps["standardscaler"].scale_[0]
una_sd["DDUlt.Riego"] = bloques["dd_riego"].named_steps["standardscaler"].scale_[0]
numericas[("", "1 sd equivale a")] = numericas.index.map(lambda c: una_sd.get(c, np.nan))
print("Lluvias: en escala logarítmica (por eso no tienen equivalencia directa en mm).")
display(numericas.round(3))

Lluvias: en escala logarítmica (por eso no tienen equivalencia directa en mm).


TCH         %Sac.Caña                        
                          coef p-valor      coef p-valor 1 sd equivale a
Lluvias (2 Meses Ant.)   0.121   0.671    -0.230   0.000             NaN
Lluvias 0 -3            -1.726   0.000     0.026   0.002             NaN
Lluvias tres a seis     -1.193   0.000    -0.010   0.254             NaN
Lluvias seis a nueve    -0.286   0.240     0.014   0.094             NaN
Luvias 9 -FC             1.761   0.000     0.109   0.000             NaN
Dosis Madurante         -0.840   0.171     0.017   0.372           0.538
DDUlt.Riego              0.620   0.035    -0.037   0.000          53.479
Edad Ult Cos            10.042   0.000     0.049   0.000           1.895
Cod.Estado #            -1.794   0.000     0.020   0.066           4.155
Dist Km                 -3.520   0.000    -0.020   0.128          18.124
anio                    -7.925   0.000     0.050   0.000           2.166
M3 Riego                 2.952   0.000    -0.010   0.244       56891.822

In [18]:
# Categóricas e indicadores significativos con mayor efecto
codificador = bloques["categoricas"].named_steps["onehotencoder"]
referencias = {v: categorias[i] for v, categorias, i in zip(CAT_REG, codificador.categories_, codificador.drop_idx_)}
print("Categoría de referencia de cada variable:")
print("  ", referencias)
print()

for t in TARGETS_REG:
    c = coefs[t]
    c = c[(c["p-valor"] < 0.05) & c["tipo"].isin(["categórica", "indicador"])]
    print(f"{t}: los 12 efectos significativos más grandes (categóricas e indicadores)")
    display(c.reindex(c["coef"].abs().sort_values(ascending=False).index).head(12).round(3))

Categoría de referencia de cada variable:
   {'Variedad': 'CC01-1228', 'Suelo': 'AMAIME', 'Zona': 'IP01', 'Producto': 'BONUS 250 EC REGULADOR FISIOLOGICO', 'T.Corte': 'MANUAL', 'Tenencia': '11', 'Cultivo': 'Normal', 'mes_cosecha': 1}

TCH: los 12 efectos significativos más grandes (categóricas e indicadores)


,tipo,coef,error est.,p-valor,IC 95% inf,IC 95% sup
Suelo_CERRITO,categórica,-22.433,4.776,0.000,-31.793,-13.073
Variedad_CC05-430,categórica,21.877,2.994,0.000,16.008,27.746
Zona_IP02,categórica,15.433,1.553,0.000,12.390,18.476
Suelo_SANTA ROSA,categórica,15.286,4.723,0.001,6.028,24.544
Variedad_CC85-92,categórica,-14.388,2.903,0.000,-20.078,-8.698
Suelo_PALMICHE,categórica,14.164,4.986,0.005,4.391,23.937
Variedad_V71-51,categórica,-14.144,3.723,0.000,-21.441,-6.847
Suelo_GALPON,categórica,13.468,4.138,0.001,5.358,21.577
Suelo_PALMERAS,categórica,13.365,4.705,0.004,4.144,22.586
Zona_IP03,categórica,12.405,1.572,0.000,9.325,15.485


%Sac.Caña: los 12 efectos significativos más grandes (categóricas e indicadores)


,tipo,coef,error est.,p-valor,IC 95% inf,IC 95% sup
Cultivo_Organico,categórica,-0.985,0.060,0.000,-1.102,-0.868
Variedad_RB73-2223,categórica,-0.759,0.083,0.000,-0.921,-0.596
Variedad_CC93-4181,categórica,0.677,0.103,0.000,0.475,0.879
mes_cosecha_10,categórica,0.664,0.035,0.000,0.596,0.733
mes_cosecha_9,categórica,0.525,0.036,0.000,0.455,0.596
Suelo_CERRITO-NIMA,categórica,-0.470,0.122,0.000,-0.708,-0.231
Variedad_CC05-430,categórica,-0.451,0.076,0.000,-0.601,-0.301
T.Corte_NO DEF.,categórica,-0.437,0.065,0.000,-0.564,-0.310
mes_cosecha_6,categórica,-0.374,0.035,0.000,-0.442,-0.305
Producto_infrequent_sklearn,categórica,-0.367,0.051,0.000,-0.468,-0.267


**Lectura.** Son asociaciones con el resto de variables constante, no relaciones de causa y efecto.

*Significancia.* De 82 coeficientes, 47 son significativos para TCH y 36 para %Sac.Caña (p < 0.05).

*TCH: numéricas.*

- **Edad al corte** es la numérica que más pesa: **+10.0 t/ha por desviación estándar**. Como una desviación son
  1.9 meses, equivale a unas 5.3 t/ha por cada mes adicional de ciclo. Signo esperado: una caña que se corta más
  madura pesa más.
- **Año:** −7.9 t/ha por desviación (2.2 años), es decir, unas 3.7 t/ha menos por año. La productividad viene
  cayendo en el periodo.
- **Distancia** (−3.5), **número de corte** (−1.8, la cepa se desgasta con cada corte) y **riego** (+3.0) tienen
  el signo esperado y son significativos.
- **Lluvias:** el signo cambia según el momento del ciclo: negativo en los primeros seis meses (−1.7 y −1.2) y
  positivo al final (+1.8). Las de 6 a 9 meses y las de los dos meses previos al corte no son significativas.
- **Dosis de madurante** no es significativa (p = 0.17).

*TCH: categóricas.* Son las de mayor magnitud: el suelo Cerrito (−22.4 t/ha), la variedad CC05-430 (+21.9) y
las zonas IP02 e IP03 (+15.4 y +12.4 frente a IP01). Hay que leerlas con cuidado: cada efecto se mide contra la
categoría de referencia (variedad CC01-1228, suelo Amaime), y los errores estándar de suelo son grandes (entre
4 y 5 t/ha). Sirven más para comparar categorías entre sí que como valor exacto.

*%Sac.Caña.*

- Dominan las categóricas: **cultivo orgánico, −0.99 puntos** frente al cultivo normal; variedad RB73-2223,
  −0.76; variedad CC93-4181, +0.68.
- **Mes de cosecha:** octubre (+0.66) y septiembre (+0.53) son los mejores frente a enero; junio resta 0.37.
- Entre las numéricas, la que más pesa es la **lluvia de los dos meses previos al corte: −0.23 puntos por
  desviación**. Llover justo antes de cosechar baja la sacarosa.
- Número de corte, distancia, riego y dosis no son significativos para la sacarosa.

En resumen: el TCH depende sobre todo de **dónde y con qué** se siembra (suelo, zona, variedad) y de **cuánto
dura el ciclo**; la sacarosa depende de **cuándo** se cosecha y del manejo cerca del corte.

### 3.4 Efecto de las semanas de madurante

La curva muestra cuánto cambia el target según las semanas entre la aplicación del madurante y el corte,
comparado con cortar a la semana 4. La banda es el intervalo de confianza del 95 %.

In [19]:
def efecto_semanas(target, referencia=4.0):
    """Curva del spline de semanas de madurante frente a la semana de referencia, con su error estándar."""
    pre = ajustados_reg[(target, "Regresión lineal")]["modelo"].named_steps["pre"]
    spline = pre.named_steps["columnas"].named_transformers_["semanas"].steps[-1][1]
    X_tr, _, _ = preparar_reg(train_reg, target)
    semanas = X_tr.loc[X_tr["Producto"].notna(), "Semanas mad."]
    grilla = np.linspace(0, semanas.clip(lower=0).quantile(0.99), 200)
    B = np.asarray(spline.transform(pd.DataFrame({"Semanas mad.": np.r_[grilla, referencia]})), dtype=float)
    B = B[:-1] - B[-1]
    cols = [c for c in coefs[target].index if c.startswith("Semanas mad._sp")]
    beta = ols[target].params[cols].to_numpy()
    cov = ols[target].cov_params().loc[cols, cols].to_numpy()
    efecto = B @ beta
    error = np.sqrt(np.einsum("ij,jk,ik->i", B, cov, B))
    # prueba conjunta: ¿todas las columnas del spline valen 0 a la vez?
    restriccion = (ols[target].params.index.to_numpy()[None, :] == np.array(cols)[:, None]).astype(float)
    return grilla, efecto, error, float(ols[target].f_test(restriccion).pvalue)


fig, ejes = plt.subplots(1, 2, figsize=(11, 3.8))
for eje, (t, unidad) in zip(ejes, [("TCH", "t/ha"), ("%Sac.Caña", "puntos")]):
    semanas, efecto, error, p = efecto_semanas(t)
    eje.fill_between(semanas, efecto - 1.96 * error, efecto + 1.96 * error, color=COLOR["azul"], alpha=0.15, lw=0)
    eje.plot(semanas, efecto, color=COLOR["azul"], lw=2)
    eje.axhline(0, color=COLOR["gris"], lw=1, ls="--")
    eje.set(title=f"{t} (p conjunto = {p:.1g})", xlabel="Semanas entre el madurante y el corte",
            ylabel=f"Cambio frente a la semana 4 ({unidad})")
    i = efecto.argmax()
    print(f"{t}: máximo en la semana {semanas[i]:.0f} ({efecto[i]:+.2f}) | semana 10: {np.interp(10, semanas, efecto):+.2f} | "
          f"semana 20: {np.interp(20, semanas, efecto):+.2f} | semana 30: {np.interp(30, semanas, efecto):+.2f}")
fig.tight_layout()
fig.savefig(FIGURAS / "reg_lineal_semanas_madurante.png", dpi=200, bbox_inches="tight")
plt.show()

TCH: máximo en la semana 0 (+1.99) | semana 10: -6.67 | semana 20: -17.02 | semana 30: -31.42
%Sac.Caña: máximo en la semana 10 (+0.60) | semana 10: +0.60 | semana 20: +0.31 | semana 30: -0.18


C:\Users\joril\AppData\Local\Temp\ipykernel_20440\584292316.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Lectura.** Las semanas de madurante son significativas en los dos modelos (p conjunto muy por debajo de 0.05),
pero con formas opuestas:

- **%Sac.Caña: sube y luego baja.** El máximo está cerca de la **semana 10** (+0.60 puntos frente a la semana 4);
  en la semana 20 todavía hay ganancia (+0.31) y hacia la semana 30 ya se perdió (−0.18).
- **TCH: baja todo el tiempo.** Frente a la semana 4 se pierden unas 6.7 t/ha en la semana 10, 17 en la semana 20
  y 31 en la semana 30.

Para el Ingenio esto es un intercambio: esperar hasta la semana 10 da la mejor sacarosa, pero cuesta tonelaje.
Una línea recta no puede representar una subida seguida de una bajada; por eso se usó el spline, y es una señal de
que en estos datos hay relaciones no lineales.

### 3.5 Valores reales frente a predichos (prueba)

In [20]:
fig, ejes = plt.subplots(1, 2, figsize=(10, 4.4))
for eje, (t, unidad) in zip(ejes, [("TCH", "t/ha"), ("%Sac.Caña", "%")]):
    r = ajustados_reg[(t, "Regresión lineal")]
    real, pred = r["y_prueba"], r["pred_prueba"]
    lo, hi = np.percentile(np.r_[real, pred], [0.2, 99.8])
    eje.scatter(real, pred, s=5, alpha=0.25, color=COLOR["azul"], linewidths=0)
    eje.plot([lo, hi], [lo, hi], color=COLOR["gris"], lw=1, ls="--")  # línea de predicción perfecta
    eje.set(xlim=(lo, hi), ylim=(lo, hi), xlabel=f"{t} real ({unidad})", ylabel=f"{t} predicho ({unidad})",
            title=f"{t}: regresión lineal (R² en prueba = {resultados_reg[(t, 'Regresión lineal')]['R2 prueba']:.3f})")
    eje.set_yticks(eje.get_xticks())
    eje.set(xlim=(lo, hi), ylim=(lo, hi))
    eje.set_aspect("equal")
    p_real, p_pred = np.percentile(real, [1, 99]), np.percentile(pred, [1, 99])
    print(f"{t}: el 98 % central de los valores reales va de {p_real[0]:.1f} a {p_real[1]:.1f}; "
          f"el de las predicciones, de {p_pred[0]:.1f} a {p_pred[1]:.1f}")
fig.tight_layout()
fig.savefig(FIGURAS / "reg_lineal_real_vs_predicho.png", dpi=200, bbox_inches="tight")
plt.show()

TCH: el 98 % central de los valores reales va de 55.7 a 208.1; el de las predicciones, de 84.9 a 176.3
%Sac.Caña: el 98 % central de los valores reales va de 9.3 a 14.9; el de las predicciones, de 10.4 a 13.7


C:\Users\joril\AppData\Local\Temp\ipykernel_20440\3724338205.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Lectura.** Los puntos siguen la diagonal, pero la nube es más "plana" que ella: el modelo **acerca todo al
promedio**. En TCH, el 98 % central de los valores reales va de 56 a 208 t/ha y el de las predicciones solo de 85
a 176. En sacarosa pasa lo mismo (9.3 a 14.9 real frente a 10.4 a 13.7 predicho). Los lotes muy malos y los muy
buenos, que son los que más le interesan al Ingenio, son los que peor predice.

**Conclusión de la sección.** La regresión lineal es un punto de referencia honesto: R² en prueba de **0.314** para
TCH y **0.370** para %Sac.Caña, sin sobreajuste y con coeficientes que tienen sentido agronómico. Su límite es
que cada variable suma su efecto por separado. No puede representar, por ejemplo, que la lluvia afecte distinto
según el suelo o la variedad. Eso es lo que se busca con los modelos de árboles de la sección 5.

## 4. Modelos de referencia: logística multinomial y KNN

Ahora el problema es de clasificación: decir si una cosecha será de nivel **Bajo, Medio o Alto** en TCH y en
sacarosa (BD_IPSA_1940). Son dos problemas separados, uno por target.

### 4.1 Cómo se evalúa cada modelo de clasificación

Igual que en regresión, todos los modelos pasan por la misma función: validación cruzada de 5 particiones
**por hacienda** en entrenamiento y después la prueba (56 haciendas que el modelo nunca vio).

Métricas:

- **Accuracy:** porcentaje de aciertos. Sola engaña cuando las clases no están balanceadas.
- **Precision, Recall y F1 (promedio de las tres clases):** cada clase pesa lo mismo, sin importar su tamaño.
- **Kappa:** compara el acierto del modelo con el que se tendría adivinando al azar. 0 es igual al azar y 1 es
  acuerdo perfecto. **Es la métrica con la que se escogen los hiperparámetros**, porque no premia irse siempre
  por la clase más grande.
- **Recall de Bajo:** de las cosechas que de verdad fueron de nivel Bajo, cuántas detecta el modelo. Es la
  métrica de negocio: no detectar un lote de bajo desempeño es el error más costoso.

In [21]:
METRICAS_CLF = {
    "Accuracy": "accuracy",
    "Precision": make_scorer(precision_score, average="macro", zero_division=0),
    "Recall": "recall_macro",
    "F1": "f1_macro",
    "Kappa": make_scorer(cohen_kappa_score),
    "Recall Bajo": make_scorer(recall_score, labels=["Bajo"], average="macro"),
}
KAPPA = METRICAS_CLF["Kappa"]
resultados_clf = {}  # (target, modelo) -> métricas. Se llena aquí y en la sección 6; se compara en la sección 7
ajustados_clf = {}   # (target, modelo) -> modelo ajustado y sus predicciones en prueba


def evaluar_clf(nombre, target, modelo):
    """Validación cruzada por hacienda en entrenamiento + ajuste con todo el entrenamiento + métricas en prueba."""
    X_tr, y_tr, g_tr = preparar_clf(train_clf, target)
    X_te, y_te, _ = preparar_clf(test_clf, target)

    cv = cross_validate(modelo, X_tr, y_tr, groups=g_tr, cv=CV_CLF, scoring=METRICAS_CLF)
    fila = {}
    for metrica in METRICAS_CLF:
        fila[f"{metrica} CV"], fila[f"{metrica} CV (sd)"] = cv[f"test_{metrica}"].mean(), cv[f"test_{metrica}"].std()

    ajustado = clone(modelo).fit(X_tr, y_tr)
    pred = ajustado.predict(X_te)
    fila["Accuracy prueba"] = accuracy_score(y_te, pred)
    fila["Precision prueba"] = precision_score(y_te, pred, average="macro", zero_division=0)
    fila["Recall prueba"] = recall_score(y_te, pred, average="macro")
    fila["F1 prueba"] = f1_score(y_te, pred, average="macro")
    fila["Kappa prueba"] = cohen_kappa_score(y_te, pred)
    fila["Recall Bajo prueba"] = recall_score(y_te, pred, labels=["Bajo"], average="macro")

    resultados_clf[(target, nombre)] = fila
    ajustados_clf[(target, nombre)] = {"modelo": ajustado, "y_prueba": y_te, "pred_prueba": pred}
    return fila


def tabla_clf():
    """Tabla con todos los modelos de clasificación evaluados hasta el momento."""
    columnas = ["Kappa CV", "Kappa CV (sd)", "Kappa prueba", "F1 CV", "F1 prueba", "Accuracy prueba",
                "Precision prueba", "Recall prueba", "Recall Bajo CV", "Recall Bajo prueba"]
    orden = [clave for t in TARGETS_CLF for clave in resultados_clf if clave[0] == t]  # agrupada por target
    tabla = pd.DataFrame(resultados_clf).T.loc[orden, columnas]
    tabla.index.names = ["target", "modelo"]
    return tabla

### 4.2 Regresión logística multinomial con regularización L2 y L1

La logística multinomial calcula una probabilidad para cada una de las tres clases. Se le agregan dos cosas:

- **`class_weight="balanced"`:** cada clase pesa según lo escasa que es, para que el modelo no descuide la clase
  Bajo (22 % de las cosechas).
- **Regularización**, que castiga los coeficientes grandes. **L2** los encoge sin anularlos; **L1** puede
  llevarlos exactamente a 0 y así descarta variables. El parámetro `C` controla cuánto se castiga: **entre más
  pequeño, más regularización**.

`C` se escoge con `GridSearchCV` entre 11 valores (de 0.001 a 100), usando la validación cruzada por hacienda y
quedándose con el mejor Kappa.

In [22]:
GRILLA_C = np.logspace(-3, 2, 11)


def logistica(**parametros):
    return Pipeline([("pre", preprocesador_clf("logistica")),
                     ("modelo", LogisticRegression(max_iter=20000, class_weight="balanced", **parametros))])


busquedas = {}  # (target, modelo) -> GridSearchCV ajustado
for t in TARGETS_CLF:
    X, y, g = preparar_clf(train_clf, t)
    candidatos = {"Logística L2": logistica(l1_ratio=0),                  # l1_ratio=0 es L2 puro
                  "Logística L1": logistica(l1_ratio=1, solver="saga")}   # l1_ratio=1 es L1 puro
    for nombre, modelo in candidatos.items():
        busqueda = GridSearchCV(modelo, {"modelo__C": GRILLA_C}, cv=CV_CLF, scoring=KAPPA).fit(X, y, groups=g)
        busquedas[(t, nombre)] = busqueda
        evaluar_clf(nombre, t, busqueda.best_estimator_)
        print(f"{t} | {nombre}: mejor C = {busqueda.best_params_['modelo__C']:.4g}")

TCH | Logística L2: mejor C = 0.1


TCH | Logística L1: mejor C = 0.1


sacarosa | Logística L2: mejor C = 0.03162


sacarosa | Logística L1: mejor C = 0.3162


### 4.3 KNN con k optimizado

KNN clasifica una cosecha según la clase de las `k` cosechas más parecidas del entrenamiento. Como trabaja con
distancias, las variables van estandarizadas y solo se usan las informativas para cada target (sección 2.3).

Se prueban dos versiones:

- **KNN:** el clásico, cada vecino vale un voto.
- **KNN corregido:** KNN no tiene `class_weight`, así que tiende a votar por las clases grandes y casi nunca
  predice Bajo. Para compararlo en igualdad de condiciones con la logística, se divide la proporción de votos de
  cada clase por la frecuencia de esa clase en entrenamiento. Además, los vecinos más cercanos pesan más
  (`weights="distance"`).

En las dos se busca `k` entre los impares de 1 a 101.

In [23]:
class KNNCorregido(ClassifierMixin, BaseEstimator):
    """KNN que divide la probabilidad de cada clase por su frecuencia en entrenamiento.

    Es el equivalente de class_weight="balanced": una clase pequeña no pierde solo por tener menos vecinos.
    Las frecuencias se calculan en fit, así que en la validación cruzada salen del entrenamiento de cada partición.
    """

    def __init__(self, n_neighbors=5, weights="uniform"):
        self.n_neighbors = n_neighbors
        self.weights = weights

    def fit(self, X, y):
        self.knn_ = KNeighborsClassifier(n_neighbors=self.n_neighbors, weights=self.weights).fit(X, y)
        self.classes_ = self.knn_.classes_
        self.frecuencias_ = np.array([np.mean(np.asarray(y) == clase) for clase in self.classes_])
        return self

    def predict_proba(self, X):
        return self.knn_.predict_proba(X)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X) / self.frecuencias_, axis=1)]


GRILLA_K = list(range(1, 102, 2))

for t in TARGETS_CLF:
    X, y, g = preparar_clf(train_clf, t)
    candidatos = {"KNN": KNeighborsClassifier(), "KNN corregido": KNNCorregido(weights="distance")}
    for nombre, clasificador in candidatos.items():
        modelo = Pipeline([("pre", preprocesador_clf("knn", target=t)), ("modelo", clasificador)])
        busqueda = GridSearchCV(modelo, {"modelo__n_neighbors": GRILLA_K}, cv=CV_CLF, scoring=KAPPA).fit(X, y, groups=g)
        busquedas[(t, nombre)] = busqueda
        evaluar_clf(nombre, t, busqueda.best_estimator_)
        print(f"{t} | {nombre}: mejor k = {busqueda.best_params_['modelo__n_neighbors']}")

TCH | KNN: mejor k = 39


TCH | KNN corregido: mejor k = 47


sacarosa | KNN: mejor k = 29


sacarosa | KNN corregido: mejor k = 23


In [24]:
# Kappa de validación cruzada según el hiperparámetro (el punto marca el valor escogido)
fig, ejes = plt.subplots(2, 2, figsize=(12, 6.5), sharey="row")
paneles = [("Logística L2", "Logística L1", "modelo__C", "C (menor = más regularización)"),
           ("KNN", "KNN corregido", "modelo__n_neighbors", "k (número de vecinos)")]
for i, t in enumerate(TARGETS_CLF):
    for j, (modelo_a, modelo_b, parametro, etiqueta) in enumerate(paneles):
        eje = ejes[i, j]
        for nombre, color in ((modelo_a, COLOR["azul"]), (modelo_b, COLOR["naranja"])):
            r = busquedas[(t, nombre)].cv_results_
            valores = np.array(r[f"param_{parametro}"], dtype=float)
            eje.plot(valores, r["mean_test_score"], color=color, lw=2, label=nombre)
            mejor = busquedas[(t, nombre)].best_index_
            eje.scatter(valores[mejor], r["mean_test_score"][mejor], color=color, s=60, zorder=3,
                        edgecolors="white", linewidths=1.5)
        if parametro == "modelo__C":
            eje.set_xscale("log")
        eje.set(title=f"{t}", xlabel=etiqueta)
        eje.legend(frameon=False)
    ejes[i, 0].set_ylabel("Kappa (validación cruzada)")
fig.tight_layout()
fig.savefig(FIGURAS / "clf_hiperparametros.png", dpi=200, bbox_inches="tight")
plt.show()

C:\Users\joril\AppData\Local\Temp\ipykernel_20440\1631382398.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.4 Desempeño: validación cruzada frente a prueba

In [25]:
display(tabla_clf().round(3))

Kappa CV  Kappa CV (sd)  Kappa prueba  F1 CV  F1 prueba  Accuracy prueba  Precision prueba  Recall prueba  Recall Bajo CV  Recall Bajo prueba
target   modelo                                                                                                                                                      
TCH      Logística L2      0.159          0.031         0.111  0.426      0.383            0.400             0.413          0.411           0.494               0.475
         Logística L1      0.167          0.021         0.093  0.428      0.384            0.395             0.390          0.405           0.494               0.465
         KNN               0.162          0.025         0.107  0.412      0.403            0.436             0.434          0.404           0.118               0.198
         KNN corregido     0.176          0.024         0.111  0.440      0.404            0.409             0.405          0.417           0.454               0.465
sacarosa Logística L2      0.242          0.021         0.231  0.476      0.464            0.509             0.473          0.485           0.626               0.436
         Logística L1      0.246          0.023         0.250  0.478      0.483            0.521             0.488          0.500           0.618               0.457
         KNN               0.278          0.021         0.237  0.510      0.470            0.530             0.522          0.475           0.348               0.223
         KNN corregido     0.302          0.024         0.289  0.526      0.523            0.546             0.525          0.526           0.613               0.468

**Lectura.**

- **Sacarosa se clasifica mejor que TCH.** En sacarosa el Kappa de prueba va de 0.23 a 0.29 y la exactitud de
  0.51 a 0.55. En TCH el Kappa de prueba va de 0.09 a 0.11 y la exactitud de 0.40 a 0.44, apenas por encima de
  lo que se lograría diciendo siempre "Medio" (39 % de las cosechas de prueba).
- **Mejor modelo de referencia en sacarosa: KNN corregido** (k = 23), con el mejor Kappa tanto en validación
  cruzada (0.302) como en prueba (0.289) y F1 de 0.52.
- **En TCH los cuatro modelos están empatados.** Sus Kappa de validación cruzada (0.159 a 0.176) difieren menos
  que la desviación entre particiones (0.02 a 0.03), así que no hay un ganador claro.
- **La corrección de KNN es necesaria.** Sin ella, KNN detecta solo el 20 % (TCH) y el 22 % (sacarosa) de las
  cosechas de nivel Bajo; con ella sube a 47 %, al nivel de la logística (44 % a 48 %). Sin esa corrección la comparación
  entre KNN y la logística no era justa.
- **L1 y L2 dan casi lo mismo** (0.167 y 0.159 en TCH; 0.246 y 0.242 en sacarosa). Regularizar ayuda poco porque
  hay pocas columnas (26) para 1 749 cosechas.
- **En TCH la prueba sale peor que la validación cruzada** en los cuatro modelos, y en una cantidad parecida
  (0.05 a 0.07 de Kappa). Como les pasa a todos, parece que las 56 haciendas de prueba son más difíciles que el
  promedio, más que un problema de un modelo en particular. En sacarosa las dos cifras coinciden.

Las curvas de la figura son bastante planas: el Kappa cambia poco en un rango amplio de `C` y de `k`. La única
caída fuerte es la de L1 con `C` muy pequeño, donde la regularización anula todos los coeficientes.

*Nota:* `C` y `k` se escogieron con la misma validación cruzada que se reporta, así que ese valor es un poco
optimista. La prueba no participó en ninguna decisión.

### 4.5 Importancia y selección de variables con L1

Con L1 se pueden ver dos cosas: **qué tanto pesa cada variable** (el tamaño de sus coeficientes) y **en qué orden
salen del modelo** cuando se aumenta la regularización. Una variable que sale primero aporta poco; la que
aguanta hasta el final es la más importante.

Las variables categóricas y la de semanas (spline) ocupan varias columnas, así que para cada variable se toma el
coeficiente más grande en valor absoluto entre sus columnas y las tres clases.

In [26]:
VARIABLES_CLF = ["lluvias", FLAG_LLUVIA, "dosismad", "edad", "cortes", "anio", "semsmad", "grupo_tenencia", "mes_cosecha"]


def variable_de(columna):
    """Variable original a la que pertenece una columna del modelo (por ejemplo, mes_cosecha_10 -> mes_cosecha)."""
    return next(v for v in VARIABLES_CLF if columna == v or columna.startswith(v + "_"))


def coeficientes(pipeline):
    """Coeficientes de la logística: una fila por columna del modelo y una columna por clase."""
    columnas = pipeline.named_steps["pre"].get_feature_names_out()
    clasificador = pipeline.named_steps["modelo"]
    return pd.DataFrame(clasificador.coef_, index=clasificador.classes_, columns=columnas).T[CLASES]


def importancia(pipeline):
    c = coeficientes(pipeline)
    return c.abs().groupby([variable_de(col) for col in c.index]).max().max(axis=1)


coef_l1 = {t: coeficientes(ajustados_clf[(t, "Logística L1")]["modelo"]) for t in TARGETS_CLF}
importancia_l1 = pd.DataFrame({t: importancia(ajustados_clf[(t, "Logística L1")]["modelo"]) for t in TARGETS_CLF})
print("Importancia: mayor |coeficiente| de cada variable en la logística L1 escogida")
display(importancia_l1.sort_values("sacarosa", ascending=False).round(3))

for t in TARGETS_CLF:
    n_cero = int((coef_l1[t] == 0).all(axis=1).sum())
    print(f"{t}: {n_cero} de {len(coef_l1[t])} columnas quedan con coeficiente 0 en las tres clases")

# Orden de salida: se aumenta la regularización (C cada vez más pequeño) y se anota cuándo se anula cada variable
CAMINO_C = [0.3162, 0.1, 0.0562, 0.0316, 0.0178, 0.01]
for t in TARGETS_CLF:
    X, y, _ = preparar_clf(train_clf, t)
    activas, salida = set(VARIABLES_CLF), {}
    for C in CAMINO_C:
        modelo = logistica(l1_ratio=1, solver="saga", C=C).fit(X, y)
        siguen = set(importancia(modelo)[lambda s: s > 0].index)
        for variable in activas - siguen:
            salida[variable] = C
        activas = siguen
    orden = sorted(salida, key=lambda v: -salida[v]) + sorted(activas)
    print(f"\n{t}: orden de salida (C en el que la variable se anula; '-' = sigue en el modelo con C = 0.01)")
    print("   " + "  ->  ".join(f"{v} ({salida.get(v, '-')})" for v in orden))

Importancia: mayor |coeficiente| de cada variable en la logística L1 escogida


,TCH,sacarosa
semsmad,0.060,1.628
mes_cosecha,0.090,1.344
lluvias,0.057,0.195
edad,0.071,0.136
grupo_tenencia,0.560,0.112
anio,0.113,0.112
dosismad,0.063,0.079
cortes,0.476,0.076
falta_dato_lluvia,0.066,0.019


TCH: 16 de 26 columnas quedan con coeficiente 0 en las tres clases
sacarosa: 5 de 26 columnas quedan con coeficiente 0 en las tres clases



TCH: orden de salida (C en el que la variable se anula; '-' = sigue en el modelo con C = 0.01)
   falta_dato_lluvia (0.0562)  ->  mes_cosecha (0.0562)  ->  semsmad (0.0562)  ->  dosismad (0.0316)  ->  lluvias (0.0178)  ->  anio (0.01)  ->  edad (0.01)  ->  grupo_tenencia (0.01)  ->  cortes (-)



sacarosa: orden de salida (C en el que la variable se anula; '-' = sigue en el modelo con C = 0.01)
   falta_dato_lluvia (0.1)  ->  grupo_tenencia (0.0562)  ->  dosismad (0.0316)  ->  anio (0.0178)  ->  edad (0.0178)  ->  cortes (0.0178)  ->  semsmad (0.0178)  ->  mes_cosecha (0.01)  ->  lluvias (-)


In [27]:
# Coeficientes por clase de las variables de una sola columna (numéricas estandarizadas e indicador)
simples = ["lluvias", "edad", "cortes", "anio", "dosismad", FLAG_LLUVIA]
display(pd.concat({t: coef_l1[t].loc[simples] for t in TARGETS_CLF}, axis=1).round(3))

fig, ejes = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1.1, 1]})
nombres = {"grupo_tenencia": "grupo de tenencia", "cortes": "número de corte", "anio": "año", "mes_cosecha": "mes de cosecha",
           "edad": "edad", FLAG_LLUVIA: "falta dato de lluvia", "dosismad": "dosis de madurante",
           "semsmad": "semanas de madurante", "lluvias": "lluvia final"}
imp = importancia_l1.rename(index=nombres).sort_values("sacarosa")
posicion = np.arange(len(imp))
ejes[0].barh(posicion + 0.2, imp["TCH"], height=0.36, color=COLOR["azul"], label="TCH")
ejes[0].barh(posicion - 0.2, imp["sacarosa"], height=0.36, color=COLOR["naranja"], label="sacarosa")
ejes[0].set_yticks(posicion, imp.index)
ejes[0].set(title="Importancia en la logística L1", xlabel="mayor |coeficiente| de la variable")
ejes[0].grid(axis="y", visible=False)
ejes[0].legend(frameon=False, loc="lower right")

mes = coef_l1["sacarosa"].filter(like="mes_cosecha_", axis=0)["Alto"]
mes.index = [int(c.split("_")[-1]) for c in mes.index]
mes = mes.reindex(range(1, 13), fill_value=0.0)  # enero es la referencia: su coeficiente es 0
ejes[1].bar(range(12), mes.values, color=COLOR["azul"], width=0.65)
ejes[1].axhline(0, color=COLOR["gris"], lw=1)
ejes[1].set_xticks(range(12), list("EFMAMJJASOND"))
ejes[1].set(title="Sacarosa Alta según el mes de cosecha", xlabel="mes de cosecha", ylabel="coeficiente frente a enero")
ejes[1].grid(axis="x", visible=False)
fig.tight_layout()
fig.savefig(FIGURAS / "clf_importancia_logistica_l1.png", dpi=200, bbox_inches="tight")
plt.show()
print("Coeficiente de sacarosa Alta por mes (enero = 0):", mes.round(2).to_dict())

TCH               sacarosa              
                    Bajo  Medio   Alto     Bajo  Medio   Alto
lluvias           -0.057  0.000  0.053    0.195  0.000 -0.151
edad               0.000 -0.071  0.062    0.136  0.000  0.000
cortes             0.147  0.000 -0.476   -0.076  0.000  0.030
anio               0.061  0.000 -0.113   -0.112  0.000  0.022
dosismad           0.000 -0.022  0.063   -0.079  0.031  0.000
falta_dato_lluvia  0.066 -0.053  0.000    0.014  0.000 -0.019

Coeficiente de sacarosa Alta por mes (enero = 0): {1: 0.0, 2: 0.0, 3: 0.61, 4: -0.56, 5: -0.71, 6: -0.81, 7: -0.8, 8: 0.0, 9: 0.66, 10: 0.87, 11: 0.18, 12: -0.17}


C:\Users\joril\AppData\Local\Temp\ipykernel_20440\790186324.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Lectura.**

*TCH.* Pesan dos variables: el **grupo de tenencia** (0.56) y el **número de corte** (0.48). Las demás quedan en
0.11 o menos. El número de corte tiene el signo esperado: más cortes bajan la probabilidad de nivel Alto (−0.48)
y suben la de Bajo (+0.15), porque la cepa se desgasta. Es también la variable más firme: es la única que
sigue en el modelo con la regularización más fuerte. L1 deja en cero 16 de las 26 columnas, y las primeras en
salir son las semanas de madurante, el mes de cosecha y el indicador de lluvia.

*Sacarosa.* Dominan las **semanas de madurante** (1.63) y el **mes de cosecha** (1.34), seguidas de lejos por la
lluvia final (0.20). Cosechar en marzo, septiembre u octubre favorece la sacarosa Alta (+0.61, +0.66 y +0.87
frente a enero); de abril a julio la reduce (−0.56 a −0.81). Más lluvia al final del ciclo sube la probabilidad
de nivel Bajo (+0.20) y baja la de Alto (−0.15). Aquí L1 solo anula 5 columnas; las últimas variables en salir
son la lluvia y el mes de cosecha, y las primeras el indicador de lluvia y el grupo de tenencia.

*Cuidado con la comparación.* Para variables de varias columnas (mes, tenencia, semanas) se toma el coeficiente
más grande, así que tienden a verse más importantes solo por tener más columnas. Por eso se mira también el
orden de salida: las semanas de madurante tienen el coeficiente más grande en sacarosa, pero no son las últimas
en salir.

*Coincide con la regresión* (sección 3): la sacarosa depende de **cuándo se cosecha** y del madurante; el TCH
depende del **desgaste de la cepa** y de quién maneja el lote (tenencia).

### 4.6 Matrices de confusión en prueba

Cada fila es la clase real y cada columna la clase que predijo el modelo. Los porcentajes son por fila, así que
la diagonal es el recall de cada clase.

In [28]:
MODELOS_REF_CLF = ["Logística L2", "Logística L1", "KNN", "KNN corregido"]
fig, ejes = plt.subplots(2, 4, figsize=(15, 7.5))
filas = []
for i, t in enumerate(TARGETS_CLF):
    for j, nombre in enumerate(MODELOS_REF_CLF):
        r = ajustados_clf[(t, nombre)]
        eje = ejes[i, j]
        matriz = confusion_matrix(r["y_prueba"], r["pred_prueba"], labels=CLASES)
        porcentaje = matriz / matriz.sum(axis=1, keepdims=True)  # por fila: la diagonal es el recall de cada clase
        eje.imshow(porcentaje, cmap="Blues", vmin=0, vmax=1)     # misma escala de color en todas las matrices
        for fila in range(3):
            for columna in range(3):
                eje.text(columna, fila, f"{porcentaje[fila, columna]:.0%}", ha="center", va="center",
                         color="white" if porcentaje[fila, columna] > 0.55 else "black")
        eje.set_xticks(range(3), CLASES)
        eje.set_yticks(range(3), CLASES)
        eje.grid(False)
        eje.set(title=f"{t}: {nombre}\nKappa = {resultados_clf[(t, nombre)]['Kappa prueba']:.3f}",
                xlabel="clase predicha", ylabel="clase real" if j == 0 else "")
        filas.append({"target": t, "modelo": nombre, **dict(zip(CLASES, porcentaje.diagonal().round(3)))})
fig.tight_layout()
fig.savefig(FIGURAS / "clf_referencia_matrices_confusion.png", dpi=200, bbox_inches="tight")
plt.show()

print("Recall por clase en prueba:")
display(pd.DataFrame(filas).set_index(["target", "modelo"]))

Recall por clase en prueba:


C:\Users\joril\AppData\Local\Temp\ipykernel_20440\1437841845.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Bajo  Medio   Alto
target   modelo                            
TCH      Logística L2   0.475  0.206  0.551
         Logística L1   0.465  0.224  0.527
         KNN            0.198  0.512  0.503
         KNN corregido  0.465  0.306  0.479
sacarosa Logística L2   0.436  0.238  0.783
         Logística L1   0.457  0.275  0.766
         KNN            0.223  0.412  0.788
         KNN corregido  0.468  0.394  0.717

**Lectura.**

- **La clase Medio es la más difícil.** La logística solo reconoce entre el 21 % y el 28 % de las cosechas de
  nivel Medio; KNN corregido llega al 31 % (TCH) y al 39 % (sacarosa). Es esperable: Medio es una franja angosta
  (25 t/ha en TCH y 0.8 puntos en sacarosa) entre las otras dos clases, y muchas cosechas quedan cerca del borde.
- **En sacarosa, la clase Alto se reconoce bien** (72 % a 79 %) y casi nunca se confunde con Bajo (1 % a 9 %).
- **En TCH hay confusiones entre los extremos:** cerca de un tercio de las cosechas de nivel Bajo se predicen
  como Alto. Con las variables disponibles antes del corte, el TCH de una hacienda nueva es difícil de anticipar.
- **KNN sin corrección** casi no predice Bajo (20 % y 22 %): manda esas cosechas a Medio y Alto.

**Conclusión de la sección.** Los modelos de referencia dejan estas cifras de Kappa en prueba para superar:

| Target | Mejor modelo de referencia | Kappa validación cruzada | Kappa prueba |
|---|---|---|---|
| TCH | Los cuatro empatados (KNN corregido y logística L2 arriba) | 0.159 a 0.176 | 0.093 a 0.111 |
| Sacarosa | KNN corregido (k = 23) | 0.302 | 0.289 |

La logística L1 queda como el modelo para **interpretar** (sus coeficientes se pueden leer) y KNN corregido como
el mejor para **predecir** sacarosa. La elección final se hace en la sección 7, después de probar los modelos
avanzados.

## 5. Modelos avanzados de regresión

_Pendiente._ Por definir con el grupo (al menos dos entre Random Forest, XGBoost y SVM).

## 6. Modelos avanzados de clasificación

_Pendiente._ Los mismos algoritmos, para los niveles de TCH y sacarosa.

## 7. Comparación y selección del mejor modelo

_Pendiente._ Tablas de validación cruzada frente a prueba y elección justificada.

## 8. Interpretabilidad e insights para el Ingenio

_Pendiente._ Importancia de variables y lectura de negocio.

## 9. Conclusiones

_Pendiente._